## Overview

どうやら`daily - (social + gaming + work)`がいい感じにAUCに貢献するらしいので試してみる。

ちゅーにんぐやらを試してみる。

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [2]:
data = pd.concat([train_df, test_df])
data['labelless_hours'] = data['daily_screen_time_hours'] - (data['gaming_hours'] + data['social_media_hours'] + data['work_study_hours'])

In [3]:
# TrainとTest分離
train_df = data.iloc[:len(train_df)].copy()
test_df = data.iloc[len(train_df):].copy()

from lightgbm import LGBMClassifier

# lightGBM
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "binary_error",
    "num_iterations": 1000,
    "learning_rate": 0.02,
    "num_leaves": 255,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "min_sum_hessian_in_leaf": 1e-3,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "feature_fraction": 0.9,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "random_state": 42,
    "verbosity": -1
}

lgbm = LGBMClassifier(**params)

In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

features = ['daily_screen_time_hours', 'social_media_hours', 'work_study_hours', 'weekend_screen_time', 'labelless_hours']

X = train_df[features]
y = train_df['addicted_label']

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_lgbm = cross_val_score(
  lgbm,
  X,
  y,
  cv=cv,
  scoring="roc_auc"
)

float(round(auc_lgbm.mean() * 100, 2))

94.2

AUCのときは`predict`じゃなくて`predict_proba`を使うぞ。

In [5]:
# 提出ファイル作成
lgbm.fit(X, y)
test_pred = lgbm.predict_proba(test_df[features])[:, 1]

submission = pd.DataFrame({
  "id": test_df['id'],
  "addicted_label": test_pred
})

In [6]:
submission.to_csv('../submissions/submission.csv', index=False)

In [7]:
submission.head()

,id,addicted_label
0,691369,0.995743
1,691370,0.905270
2,691371,0.981134
3,691372,0.979635
4,691373,0.996582


In [8]:
submission.shape

(296302, 2)

In [9]:
submission.isna().sum()

id                0
addicted_label    0
dtype: int64